# 03 — Preprocesamiento y segmentación pulmonar validada

## Objetivo

Aplicar segmentación (`src/lung_segmentation.py`) que combina
máscara corporal, contraste local multi-escala, separación izquierda/derecha,
envolvente convexa y un margen perilesional. Se valida de forma cuantitativa
con las **1.211 cajas de lesión** anotadas en TBX11K.

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
from src.artifacts import ensure_artifacts
ensure_artifacts(
    [DATA / "df_model.csv"],
    [sys.executable, "scripts/build_model_meta.py"],
    "Metadatos de modelado (8.256 imágenes).",
)
ensure_artifacts(
    [DATA / "seg_summary_v2.json", DATA / "seg_quality_v2.csv",
     REP / "03_segmentacion_ejemplos.png", REP / "03_segmentacion_lesiones.png",
     REP / "03_segmentacion_comparacion.png", REP / "03_segmentacion_distribucion.png"],
    [sys.executable, "scripts/validate_segmentation.py"],
    "Validación de segmentación con cajas de lesión.",
)


## 1. Dataset base (`df_base.csv`, generado en 01)


In [ ]:
base = pd.read_csv(DATA / "df_base.csv")
print("Imágenes etiquetadas (df_base, 01):", len(base))
print(base["class_name"].value_counts().to_string())
print("\nLa deduplicación exacta para modelar (8.256) se hace en 05 "
      "y se guarda en df_model.csv.")


## 2. Segmentación

1. **Máscara corporal**: se eliminan las componentes oscuras conectadas al
   borde (fondo, marcos, colimación) y se conserva el torso.
2. **Score de oscuridad local**: `gaussiano(σ=24) − imagen`; positivo en
   regiones más oscuras que su entorno amplio (pulmones), robusto a la
   iluminación global.
3. **Separación izquierda/derecha** y eliminación del mediastino central.
4. **Envolvente convexa + margen de 20 px**: absorbe opacidades, lesiones
   periféricas y apicales que un umbral de intensidad dejaría fuera.
5. **Control de calidad**: área, balance izquierda-derecha, contraste y
   bandera de respaldo. Si falla, se usa un campo torácico grueso marcado
   como `fallback`.

La máscara resultante es un **campo pulmonar con margen**, no una
segmentación clínica de parénquima. Es deliberadamente generosa para no
eliminar lesiones; se reporta como tal.

## 3. Validación contra las cajas de lesión

In [ ]:
summary = json.loads((DATA / "seg_summary_v2.json").read_text(encoding="utf-8"))
rows = []
for name, key in [("Otsu anterior", "old"), ("Segmentación validada", "new")]:
    rows.append({
        "Método": name,
        "Cobertura media": summary[key]["coverage_mean"],
        "Cobertura ≥ 0.8": summary[key]["coverage_ge_0.8"],
        "Cobertura ≥ 0.5": summary[key]["coverage_ge_0.5"],
        "Centroide dentro": summary[key]["centroid_inside"],
    })
pd.DataFrame(rows).round(3)

In [ ]:
display(Image(filename=str(REP / "03_segmentacion_comparacion.png")))

La cobertura media de las cajas pasa de ~0.54 a ~0.91, y la proporción de
imágenes con cobertura ≥ 0.8 pasa de ~15 % a ~89 %. Además, el centroide de
la lesión queda dentro de la máscara en ~99 % de los casos. Esto corrige el
fallo crítico del pipeline anterior.

## 4. Ejemplos por clase

In [ ]:
display(Image(filename=str(REP / "03_segmentacion_ejemplos.png")))

## 5. Lesiones: peores y mejores casos

In [ ]:
display(Image(filename=str(REP / "03_segmentacion_lesiones.png")))

## 6. Calidad por clase

In [ ]:
q = pd.read_csv(DATA / "seg_quality_v2.csv")
q.groupby("class_name").agg(
    area=("area", "mean"), balance=("balance", "mean"),
    contraste=("contrast", "mean"), fallback=("fallback", "mean")
).round(3)

In [ ]:
display(Image(filename=str(REP / "03_segmentacion_distribucion.png")))

**Lectura:** el área media es ~0.49 del cuerpo, con diferencias pequeñas
entre clases (health ~0.49, sick ~0.46, tb ~0.52). Esa diferencia es una
posible señal residual de forma; por eso el modelado incluye un baseline de
**solo forma** y otro de **solo contexto/fondo**.

## 7. Preprocesamiento: original → bilateral → CLAHE → recorte

In [ ]:
from src.preprocessing import preprocess_steps, preprocess_with_mask

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, cls in enumerate(["health", "sick", "tb"]):
    row = base[base["class_name"] == cls].sample(1, random_state=11).iloc[0]
    steps = preprocess_steps(row["image_path"])
    img, mask, q = preprocess_with_mask(row["image_path"], modo="thorax")
    axes[i, 0].imshow(steps["original"], cmap="gray"); axes[i, 0].set_title(f"{cls} original")
    axes[i, 1].imshow(steps["bilateral"], cmap="gray"); axes[i, 1].set_title("bilateral")
    axes[i, 2].imshow(steps["clahe"], cmap="gray"); axes[i, 2].set_title("CLAHE clip=2")
    axes[i, 3].imshow(img, cmap="gray"); axes[i, 3].set_title(f"recorte torácico | área={q.area_fraction:.2f}")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()

## Conclusiones

- La segmentación anterior no era neutra: excluía las lesiones y además
  introducía el contorno de la máscara como posible atajo.
- La nueva segmentación **incluye las lesiones** (cobertura media 0.91) y
  mantiene un control de calidad explícito.
- El pipeline ya no enmascara con ceros: el modo principal es un recorte
  torácico que conserva la textura original.
- Persisten limitaciones: la máscara es generosa (incluye pared torácica),
  no es una segmentación clínica de parénquima y no se dispone de máscaras
  manuales de pulmón para calcular IoU. La validación con cajas de lesión es
  la evidencia disponible y es cuantitativa.

**Siguiente etapa:** `04_Analisis_Regional.ipynb`.